# verify05: 頭痛3質問を1つのBERTが学習（遷移なし・各質問の正答率）

verify04 の **B（遷移BERT）から 遷移（決定木トラバーサル）・履歴・トリアージを外した版**。
**頭痛の3質問ノード（痛み/しびれ/異常言動）を1本のBERTがまとめて学習**し、質問ごとの正答率を見る。

| 観点 | verify04 (B0/B1) | **verify05（本notebook）** |
|---|---|---|
| モデル | 共有BERT + 遷移でトリアージ | **共有BERT 1本のみ（遷移なし）** |
| 入力 | 質問+採用ペア(+履歴) | **各ペア（相談員Q＋通報者A）そのもの** |
| ラベル | はい/いいえ/不明(3値) | **はい/いいえ/不明/非該当（0/1/2/-1）** |
| 評価 | ノード別 + トリアージ | **各質問ノードの正答率のみ（トリアージなし）** |

- データ: `dataset/headache_symptom_pair_conversations_202607081836.csv`（3600ペア＝各ノード1200, 4ラベル×300で均衡）
- ラベル: `0=はい / 1=いいえ / 2=不明 / -1=非該当（その質問の答えになっていない別質問のペア）`
- 5-fold CV（node×label層化）で全ペアのOOF予測を作り、質問ノードごとに正答率を集計。
- 学習ループ等は verify04 / painful を流用（推論前 `eval()` 固定も踏襲）。

> 非該当(-1)を含めると「その発話が本当にその質問の答えか（relevance）」まで問う4値問題になる。
> `INCLUDE_NONAPPLICABLE=False` にすれば はい/いいえ/不明 の3値だけに絞れる。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess, glob

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules

HEADACHE_CSV = 'headache_symptom_pair_conversations_202607081836.csv'


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', HEADACHE_CSV)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', HEADACHE_CSV)
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

# 2. インポート & 設定

In [ ]:
import time, random
from typing import Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import transformers
transformers.logging.set_verbosity_error()   # 余計なロードレポート/警告を抑制

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# ===== 設定 =====
BASE_MODEL = 'cl-tohoku/bert-base-japanese-v3'   # 差し替えで別モデル（例: sbintuitions/modernbert-ja-130m）
MAX_LENGTH = 128            # ペアは短いので128で十分
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
N_FOLDS = 5
SEED = 42

INCLUDE_NONAPPLICABLE = True   # -1（非該当）を含めるか。Falseで はい/いいえ/不明 の3値のみ
SAMPLE_PER_NODE = None         # CPUで速く回したいとき: 例 150（各ノードから150件だけ）。None=全件


def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'BASE_MODEL={BASE_MODEL} MAX_LENGTH={MAX_LENGTH} epochs={NUM_EPOCHS} folds={N_FOLDS} '
      f'include_-1={INCLUDE_NONAPPLICABLE} sample/node={SAMPLE_PER_NODE}')

# 3. データ読み込み：頭痛3ノードのペアを1つの表に

各行が1ペア（相談員Q＋通報者A）。`is_*` でどのノードか、`label_*` でそのノードの正解が決まる。
入力はペアのテキストそのもの（遷移・履歴なし）。

In [ ]:
NODES = [
    {'key': 'sudden_severe', 'flag': 'is_sudden_severe', 'lab': 'label_sudden_severe', 'jp': '痛み（突然の激痛か）'},
    {'key': 'numbness',      'flag': 'is_numbness',      'lab': 'label_numbness',      'jp': 'しびれ／麻痺'},
    {'key': 'behavior',      'flag': 'is_behavior',      'lab': 'label_behavior',      'jp': '異常な言動・行動'},
]
JP = {n['key']: n['jp'] for n in NODES}
LABEL_MEANING = {'0': 'はい', '1': 'いいえ', '2': '不明', '-1': '非該当'}

d = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

def _node_of(r):
    for n in NODES:
        if bool(r[n['flag']]):
            return n['key']
    return None

_labcol = {n['key']: n['lab'] for n in NODES}
recs = []
for _, r in d.iterrows():
    nd = _node_of(r)
    if nd is None:
        continue
    code = str(r[_labcol[nd]])
    if (not INCLUDE_NONAPPLICABLE) and code == '-1':
        continue
    recs.append({'row_id': r['ID'], 'node': nd, 'text': str(r['ペア']), 'code': code})
ex = pd.DataFrame(recs)

if SAMPLE_PER_NODE:
    ex = (ex.groupby('node', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_NODE), random_state=SEED))
            .reset_index(drop=True))

le = LabelEncoder()
ex['label'] = le.fit_transform(ex['code'])
NUM_LABELS = len(le.classes_)
print('ペア数:', len(ex), '/ ノード:', ex['node'].nunique(), '/ ラベル:', list(le.classes_),
      '→ NUM_LABELS =', NUM_LABELS)
print('ノード×ラベル件数:')
display(pd.crosstab(ex['node'], ex['code']))
display(ex.head(4))

# 4. 学習・モデル部品（verify04 / painful 流用）

学習ループ・`eval()`固定（推論前にdropoutを切る）は verify04 と同じ。出力ヘッドは NUM_LABELS。

In [ ]:
_tok_cache: Dict[str, 'AutoTokenizer'] = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


class PairDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts, self.labels = list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=NUM_LABELS,
                                                               trust_remote_code=True)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        print(f'  [警告] 語彙数{len(tok)}!=vocab{model.config.vocab_size} → resize')
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


def train_model(texts, labels):
    set_seed(SEED)
    tok = get_tokenizer(BASE_MODEL)
    model = build_model(BASE_MODEL)
    loader = DataLoader(PairDataset(texts, labels, tok, MAX_LENGTH),
                        batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(NUM_EPOCHS):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    model.eval()   # ★推論前にevalへ（dropoutを切る）
    return model, tok


@torch.no_grad()
def predict_texts(model, tok, texts):
    model.eval()
    preds = []
    texts = list(texts)
    for i in range(0, len(texts), EVAL_BATCH_SIZE):
        chunk = texts[i:i + EVAL_BATCH_SIZE]
        enc = tok(chunk, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return preds


print('学習部品を定義（verify04/painful流用・eval固定）')

# 5. 5-fold CV（node×label 層化）で全ペアのOOF予測

CPUでの全件学習は重い（3600ペア×3ep×5fold＝数十分〜）。速く確認したいときは
上の `SAMPLE_PER_NODE=150` などにするか、`N_FOLDS` を小さく。

In [ ]:
strat = (ex['node'].astype(str) + '_' + ex['code'].astype(str)).values
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

ex = ex.reset_index(drop=True)
ex['pred'] = -99
for fold, (tr, te) in enumerate(skf.split(ex.index, strat)):
    t0 = time.time()
    model, tok = train_model(ex.loc[tr, 'text'].tolist(), ex.loc[tr, 'label'].tolist())
    preds = predict_texts(model, tok, ex.loc[te, 'text'].tolist())
    ex.loc[te, 'pred'] = preds
    print(f'  fold{fold}: train={len(tr)} test={len(te)}  ({time.time()-t0:.0f}s)')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert (ex['pred'] != -99).all(), 'OOF未充填の例がある'
print('OOF予測 完了')

# 6. 結果：各質問での正答率（トリアージなし）

In [ ]:
ex['correct'] = (ex['pred'] == ex['label']).astype(int)
overall_acc = ex['correct'].mean()
overall_f1 = f1_score(ex['label'], ex['pred'], average='macro', zero_division=0)
print(f'=== 全体 ===  accuracy={overall_acc:.3f}  macro-F1={overall_f1:.3f}  (n={len(ex)})')

# --- 各質問ノードでの正答率 ---
node_tbl = (ex.groupby('node')
              .agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
              .reset_index())
node_tbl.insert(1, '質問', node_tbl['node'].map(JP))
node_tbl['正答率'] = node_tbl['正答率'].map(lambda x: f'{x:.3f}')
print('\n===== 各質問ノードでの正答率 =====')
display(node_tbl)

# --- ノード×ラベル別の正答率（どのラベルで外すか）---
ex['正解ラベル'] = ex['code'].map(LABEL_MEANING)
lab_tbl = (ex.groupby(['node', '正解ラベル'])
             .agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
             .reset_index())
lab_tbl['正答率'] = lab_tbl['正答率'].map(lambda x: f'{x:.3f}')
print('\n===== ノード×正解ラベル別 正答率 =====')
display(lab_tbl)

node_tbl.to_csv(os.path.join(OUT_DIR, 'verify05_headache_node_acc.csv'), index=False, encoding='utf-8-sig')
lab_tbl.to_csv(os.path.join(OUT_DIR, 'verify05_headache_label_acc.csv'), index=False, encoding='utf-8-sig')
print('\nsaved: output/verify05_headache_node_acc.csv, output/verify05_headache_label_acc.csv')

# 7. まとめ（読み方）

- **1本のBERTが頭痛3質問をまとめて学習**したときの、質問ごとの正答率。専用BERT（for_share）や遷移(verify04)と比較する土台。
- `-1=非該当` を含む4値なので、「別質問のペアを混同せず弾けるか（relevance）」も評価に入る。`INCLUDE_NONAPPLICABLE=False` で純粋な はい/いいえ/不明 の3値に切替可。
- ノード×ラベル別の表で、どのラベル（特に不明/非該当）で落ちているかが分かる。
- 遷移・履歴・トリアージは一切なし。各ペアは独立に「テキスト ⇒ ラベル」を当てるだけ。